In [ ]:
!pip install -U diffusers transformers accelerate torch torchvision -q
!pip uninstall -y torchaudio -q

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("تم تسجيل الدخول ✅")

تم تسجيل الدخول ✅


In [3]:
import requests
url = "https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json"
briefs = requests.get(url).json()
print(f"عدد البريفات: {len(briefs)}")

عدد البريفات: 30


In [ ]:
!pip install -U bitsandbytes -q

In [4]:
import torch, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffusersBnBConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnBConfig

model_id = "black-forest-labs/FLUX.1-schnell"

# تكميم الـ transformer (أكبر جزء بالموديل)
transformer_quant_config = DiffusersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
transformer = FluxTransformer2DModel.from_pretrained(
    model_id, subfolder="transformer",
    quantization_config=transformer_quant_config,
    dtype=torch.bfloat16,
)

# تكميم الـ T5 text encoder (تاني أكبر جزء)
text_encoder_quant_config = TransformersBnBConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
text_encoder_2 = T5EncoderModel.from_pretrained(
    model_id, subfolder="text_encoder_2",
    quantization_config=text_encoder_quant_config,
    dtype=torch.bfloat16,
)

# تجميع البايبلاين بالأجزاء المكمّمة
pipe = FluxPipeline.from_pretrained(
    model_id,
    transformer=transformer,
    text_encoder_2=text_encoder_2,
    dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

print("✅ الموديل جاهز بنسخة مكمّمة")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:208: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✅ الموديل جاهز بنسخة مكمّمة


In [5]:
def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, {', '.join(b['visual_style'])} style, vector logo, clean background, no text"

test_briefs = briefs[:5]

for brief in test_briefs:
    prompt = brief_to_image_prompt(brief)
    image = pipe(prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
    filename = f"FLUX_{brief['id']}.png"
    image.save(filename)
    print(f"✅ {brief['id']} → {filename}")

  0%|          | 0/4 [00:00<?, ?it/s]

✅ BR001 → FLUX_BR001.png


  0%|          | 0/4 [00:00<?, ?it/s]

✅ BR002 → FLUX_BR002.png


  0%|          | 0/4 [00:00<?, ?it/s]

✅ BR003 → FLUX_BR003.png


  0%|          | 0/4 [00:00<?, ?it/s]

✅ BR004 → FLUX_BR004.png


  0%|          | 0/4 [00:00<?, ?it/s]

✅ BR005 → FLUX_BR005.png
